# RLVR TerminalBench — Qwen2.5-3B on Colab

GRPO training with Qwen2.5-3B-Instruct on terminalbench tasks.
Works on free Colab T4 GPU (~6GB VRAM needed).

**Runtime:** Go to Runtime > Change runtime type > **T4 GPU**

## 1. Install dependencies + clone repo

In [ ]:
!pip install -q transformers trl accelerate torch datasets pyyaml tqdm

In [ ]:
import os

REPO_DIR = '/content/AfterQuery'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/NathanG2022/AfterQuery.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

## 2. Check runtime

In [ ]:
import torch

print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('No GPU detected — running on CPU (slower but works for TinyLlama)')
    DEVICE = 'cpu'

## 3. Configuration

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

# Training
NUM_PROMPTS = 256
TOTAL_STEPS = 500
BATCH_SIZE = 2
NUM_GENERATIONS = 4
LEARNING_RATE = 1e-5
MAX_NEW_TOKENS = 128
MAX_STEPS_PER_EPISODE = 5

# Eval
NUM_EVAL_EPISODES = 50
SEED = 42

# Reward weights
STEP_PENALTY = 0.01
W_SUCCESS = 1.0
W_EFF = 0.1
W_QUALITY = 0.05

OUTPUT_DIR = 'models/rlvr'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Model: {MODEL_NAME}')
print(f'Steps: {TOTAL_STEPS}, Batch: {BATCH_SIZE}, Generations: {NUM_GENERATIONS}')
print(f'Prompts: {NUM_PROMPTS}, Max tokens: {MAX_NEW_TOKENS}')
print(f'Eval episodes: {NUM_EVAL_EPISODES}, Seed: {SEED}')
print(f'Output: {OUTPUT_DIR}')

## 4. Sanity check the environment

In [ ]:
import sys
sys.path.insert(0, '.')

from envs.terminalbench_client import TerminalBenchClient
from envs.terminalbench_env import TerminalBenchEnv
from envs.tasks import get_all_tasks
from envs.utils import extract_command

tasks = get_all_tasks()
print(f'Total tasks: {len(tasks)}')
for t in tasks:
    print(f'  [{t.difficulty:>6s}] {t.task_id}: {t.description[:70]}')

# Quick smoke test — run one command
client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
env = TerminalBenchEnv(client, max_steps=MAX_STEPS_PER_EPISODE,
                       step_penalty=STEP_PENALTY, w_success=W_SUCCESS,
                       w_eff=W_EFF, w_quality=W_QUALITY)
obs = env.reset()
print(f'\nSample task: {env.task.task_id}')
obs, reward, done, info = env.step('echo "hello world" > hello.txt')
print(f'Score: {info["success_score"]:.2f}, Reward: {reward:.4f}')
print('Environment OK!')

## 5. Build prompt dataset

In [ ]:
from datasets import Dataset

tb_client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
env = TerminalBenchEnv(tb_client, max_steps=MAX_STEPS_PER_EPISODE,
                       step_penalty=STEP_PENALTY, w_success=W_SUCCESS,
                       w_eff=W_EFF, w_quality=W_QUALITY)

prompts = []
for _ in range(NUM_PROMPTS):
    obs = env.reset()
    prompts.append({'prompt': obs})

train_dataset = Dataset.from_list(prompts)
print(f'Dataset: {len(train_dataset)} prompts')
print(f'\nExample prompt:\n{prompts[0]["prompt"][:300]}...')

## 6. Define reward function

In [ ]:
def make_reward_func():
    tb_client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
    env = TerminalBenchEnv(
        tb_client, max_steps=MAX_STEPS_PER_EPISODE,
        step_penalty=STEP_PENALTY, w_success=W_SUCCESS,
        w_eff=W_EFF, w_quality=W_QUALITY,
    )

    def reward_func(prompts, completions, **kwargs):
        rewards = []
        for prompt, completion in zip(prompts, completions):
            command = extract_command(completion)
            env.reset()
            _obs, reward, _done, _info = env.step(command)
            rewards.append(reward)
        return rewards

    return reward_func

reward_func = make_reward_func()
print('Reward function ready')

## 7. Train (GRPO)

In [ ]:
from trl import GRPOConfig, GRPOTrainer

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=NUM_GENERATIONS,
    num_train_epochs=1,
    max_steps=TOTAL_STEPS,
    max_completion_length=MAX_NEW_TOKENS,
    temperature=0.7,
    top_p=0.9,
    beta=0.04,
    logging_steps=10,
    save_strategy='steps',
    save_steps=100,
    report_to='none',
    bf16=use_bf16,
    fp16=use_fp16,
    use_cpu=not torch.cuda.is_available(),
    gradient_checkpointing=torch.cuda.is_available(),
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=[reward_func],
    args=grpo_config,
    train_dataset=train_dataset,
)

print(f'Training {MODEL_NAME} for {TOTAL_STEPS} steps...')
print(f'Checkpoints saved every 100 steps to {OUTPUT_DIR}')
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f'\nModel saved to {OUTPUT_DIR}')

## 8. Evaluate: trained vs base

In [ ]:
import random
from statistics import mean
from transformers import AutoTokenizer, AutoModelForCausalLM


def evaluate_model(model_path, num_episodes, seed, label='model'):
    """Evaluate a model and return per-episode results."""
    random.seed(seed)
    print(f'\nEvaluating {label} ({model_path})...')

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_path)
    model.eval()
    max_ctx = getattr(model.config, 'max_position_embeddings', 2048)

    tb_client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
    env = TerminalBenchEnv(
        tb_client, max_steps=MAX_STEPS_PER_EPISODE,
        step_penalty=STEP_PENALTY, w_success=W_SUCCESS,
        w_eff=W_EFF, w_quality=W_QUALITY,
    )

    results = []
    for ep in range(num_episodes):
        obs = env.reset()
        done = False
        episode_reward = 0.0
        last_score = 0.0
        task_id = env.task.task_id
        difficulty = env.task.difficulty

        while not done:
            inputs = tokenizer(
                obs, return_tensors='pt', truncation=True,
                max_length=max_ctx - MAX_NEW_TOKENS,
            ).to(model.device)

            with torch.no_grad():
                out_ids = model.generate(
                    inputs['input_ids'],
                    attention_mask=inputs['attention_mask'],
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True, temperature=0.7, top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                )

            new_ids = out_ids[0][inputs['input_ids'].shape[1]:]
            raw = tokenizer.decode(new_ids, skip_special_tokens=True)
            action = extract_command(raw)
            obs, reward, done, info = env.step(action)
            episode_reward += reward
            last_score = info['success_score']

        status = 'PASS' if last_score >= 1.0 else 'FAIL'
        print(f'  ep{ep+1:>2d} [{status}] {task_id:<24s} score={last_score:.2f} steps={info["step_count"]}')
        results.append({
            'task_id': task_id, 'difficulty': difficulty,
            'score': last_score, 'reward': episode_reward,
            'steps': info['step_count'],
        })

    del model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results

In [ ]:
print('=== Evaluating BASE model ===')
base_results = evaluate_model(MODEL_NAME, NUM_EVAL_EPISODES, SEED, label='base')

print('\n=== Evaluating TRAINED model ===')
trained_results = evaluate_model(OUTPUT_DIR, NUM_EVAL_EPISODES, SEED, label='trained')

## 9. Compare results

In [ ]:
def summarize(results):
    scores = [r['score'] for r in results]
    return {
        'mean_score': mean(scores),
        'success_rate': mean(1.0 if s >= 1.0 else 0.0 for s in scores),
        'mean_reward': mean(r['reward'] for r in results),
        'mean_steps': mean(r['steps'] for r in results),
    }

b = summarize(base_results)
t = summarize(trained_results)

print('=' * 60)
print('  COMPARISON: Base vs RLVR-Trained')
print('=' * 60)
print(f'  {"Metric":<20s}  {"Base":>10s}  {"Trained":>10s}  {"Delta":>10s}')
print(f'  {"-"*20}  {"-"*10}  {"-"*10}  {"-"*10}')

for key, label, fmt in [
    ('mean_score', 'Mean Score', '.3f'),
    ('success_rate', 'Success Rate', '.3f'),
    ('mean_reward', 'Mean Reward', '.3f'),
    ('mean_steps', 'Mean Steps', '.2f'),
]:
    bv, tv = b[key], t[key]
    delta = tv - bv
    sign = '+' if delta >= 0 else ''
    print(f'  {label:<20s}  {format(bv, fmt):>10s}  {format(tv, fmt):>10s}  {sign + format(delta, fmt):>10s}')

# By difficulty
print()
for diff in ['easy', 'medium', 'hard']:
    bs = [r['score'] for r in base_results if r['difficulty'] == diff]
    ts = [r['score'] for r in trained_results if r['difficulty'] == diff]
    if not bs and not ts:
        continue
    bm = mean(bs) if bs else 0.0
    tm = mean(ts) if ts else 0.0
    d = tm - bm
    sign = '+' if d >= 0 else ''
    print(f'  {diff:<10s}  score: {bm:.3f} -> {tm:.3f} ({sign}{d:.3f})  n={len(bs)}/{len(ts)}')

# Verdict
delta_score = t['mean_score'] - b['mean_score']
if delta_score > 0.01:
    verdict = 'RLVR training IMPROVED model performance'
elif delta_score < -0.01:
    verdict = 'RLVR training DECREASED model performance'
else:
    verdict = 'RLVR training had NO SIGNIFICANT EFFECT'
print(f'\n  >> {verdict}')
print()